In [1]:
import pandas as pd
import re

# File uploaded to Colab runtime
FILE_PATH = "50 Research Question (PICO format) (3).xlsx"

In [2]:
# Read the excel file into a pandas DataFrame
df_raw = pd.read_excel(FILE_PATH)

# Display first few rows to verify loading
df_raw.head()

,#,Clinical Specialization,Research Question,PICO Breakdown
0,1,Cardiology,In adults with heart failure with preserved ej...,• P: Adults with HFpEF (LVEF ≥ 50%)• I: SGLT2 ...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,• P: Adults undergoing PCI for acute coronary ...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,• P: Adults with severe asymptomatic aortic st...
3,4,Cardiology,In patients with persistent atrial fibrillatio...,• P: Adults with persistent atrial fibrillatio...
4,5,Cardiology,[Diagnostic] What is the diagnostic test accur...,• P: Adults with stable chest pain suspected o...


In [3]:
def extract_pico_components(text):
    """Extracts P, I, C, and O components using regular expression pattern matching."""
    if not isinstance(text, str):
        return pd.Series({'P': None, 'I': None, 'C': None, 'O': None})

    p_match = re.search(r'P:\s*(.*?)(?=\s*•?\s*[ICO]:|$)', text, re.DOTALL)
    i_match = re.search(r'I:\s*(.*?)(?=\s*•?\s*[PCO]:|$)', text, re.DOTALL)
    c_match = re.search(r'C:\s*(.*?)(?=\s*•?\s*[PIO]:|$)', text, re.DOTALL)
    o_match = re.search(r'O:\s*(.*?)(?=\s*•?\s*[PIC]:|$)', text, re.DOTALL)

    return pd.Series({
        'P': p_match.group(1).strip() if p_match else None,
        'I': i_match.group(1).strip() if i_match else None,
        'C': c_match.group(1).strip() if c_match else None,
        'O': o_match.group(1).strip() if o_match else None
    })

# Apply extraction across all rows
pico_df = df_raw['PICO Breakdown'].apply(extract_pico_components)

In [4]:
# Construct structured dataframe with specified columns
df_final = pd.DataFrame({
    'srno': df_raw['#'],
    'clinical_speciality': df_raw['Clinical Specialization'],
    'research_question': df_raw['Research Question'],
    'P': pico_df['P'],
    'I': pico_df['I'],
    'C': pico_df['C'],
    'O': pico_df['O']
})

# Output file destination on Colab runtime
OUTPUT_CSV = "pico_dataset.csv"
df_final.to_csv(OUTPUT_CSV, index=False)

print(f"Successfully processed {len(df_final)} rows and saved to '{OUTPUT_CSV}'.")

# Preview the restructured data
df_final.head()

Successfully processed 50 rows and saved to 'pico_dataset.csv'.


,srno,clinical_speciality,research_question,P,I,C,O
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...
3,4,Cardiology,In patients with persistent atrial fibrillatio...,Adults with persistent atrial fibrillation,Catheter ablation (radiofrequency/cryoballoon),Antiarrhythmic drug therapy (Class I or III),All-cause mortality; recurrence of atrial tach...
4,5,Cardiology,[Diagnostic] What is the diagnostic test accur...,Adults with stable chest pain suspected of CAD,Coronary Computed Tomography Angiography (CCTA),Invasive coronary angiography with FFR (Refere...,Sensitivity and specificity for obstructive CAD
